# Fine-tuning Run 2: Detector Continuation

This run starts from the selected detector from run 1 and uses the same prepared dataset. It compares the original detector and the two fine-tuned checkpoints under common evaluation settings.


## 01. Run settings

Defines the input artifacts, detector resolution, batch size, and 20-epoch schedule used for this run.


In [1]:
# 1 | Settings
from pathlib import Path
import os, sys, csv, json, time, shutil, tarfile, zipfile, tempfile
import subprocess

WORKING = Path("/kaggle/working")
# If Drive refuses a download, attach a private Kaggle dataset and set its folder here.
# Example: LOCAL_INPUT_DIR = "/kaggle/input/your-dataset-slug"
LOCAL_INPUT_DIR = None
DOWNLOAD_OPTIONAL_FILES = False

DRIVE_IDS = {
    "finetune_v4.tar": "1okdPs196qHFB4iluMfihUApFyiUomtPJ",
    "last.pt": "1xoJ6hPptMDE22ESEzS3dOtCceLxFUslC",
    "detector_v3_hardneg.pt": "1rkKXdfLgbbP7MwbFd0kbKSNM24_-7u5q",
    "config.yaml": "17Recc82iMl3oqyPRIXTLHFmUwW3HcX0t",
    "verifier.pt": "1LXl4TaTGuFNvpxDdxz_e5vPnZHP2zPHJ",
    "detector_old.pt": "12vuayUoms1Sm3aAZRjaUf9PSDRPgHbWV",
}
REQUIRED = ("finetune_v4.tar", "detector_v3_hardneg.pt", "detector_old.pt")
# v4 uses the same fire/smoke IDs as both checkpoints.
NAMES = {0: "fire", 1: "smoke"}
IMGSZ, BATCH, WORKERS, DEVICE = 800, 4, 2, 0
EVAL_BATCH = 4
EPOCHS, LR, SEED = 20, 0.0001, 20261114
THRESHOLDS = (0.20, 0.30, 0.40)
PRIMARY_CONF = 0.20
ULTRALYTICS_VERSION = "8.4.194"  # Same version that saved the Modal checkpoints.

if not WORKING.is_dir():
    raise RuntimeError("Open this notebook on Kaggle first.")
SCRATCH = Path(tempfile.mkdtemp(prefix="fire-smoke-v4-"))
INPUTS = SCRATCH / "inputs"
INPUTS.mkdir()
RUNS = WORKING / "fire_smoke_v4_kaggle"
RUNS.mkdir(exist_ok=True)
EXPORT = WORKING / "detector_v4_kaggle.pt"

## 02. Training environment

Checks the installed CUDA and PyTorch setup and loads the requested Ultralytics version.


In [2]:
# 2 | Install matching Ultralytics, keep Kaggle's existing PyTorch/CUDA
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Kaggle Settings, choose GPU T4 x2.")
print(torch.__version__, " / ", torch.cuda.get_device_name(DEVICE),
      f" / {torch.cuda.get_device_properties(DEVICE).total_memory / 2**30:.1f} GiB")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"ultralytics=={ULTRALYTICS_VERSION}", "gdown>=5,<7"], check=True)
import gdown, numpy as np, yaml
import ultralytics
from ultralytics import YOLO
assert ultralytics.__version__ == ULTRALYTICS_VERSION, "Restart the kernel and run again to load the pinned version."

2.11.0+cu130  /  Tesla T4  / 14.6 GiB
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 03. Locate training inputs

Resolves the prepared dataset and detector checkpoints from the configured input sources.


In [3]:
# 3 | Download from Drive or read attached Kaggle Dataset files
def check_file(path, filename):
    if not path.is_file() or not path.stat().st_size:
        raise ValueError("Missing or empty file")
    if filename.endswith(".tar") and not tarfile.is_tarfile(path):
        raise ValueError("Not a readable tar archive")
    if filename.endswith(".pt") and not zipfile.is_zipfile(path):
        raise ValueError("Not a PyTorch ZIP checkpoint (check that Drive did not return an HTML page)")
    if filename.endswith(".yaml"):
        obj = yaml.safe_load(path.read_text())
        if not isinstance(obj, dict) or "models" not in obj:
            raise ValueError("Expected the project configuration YAML")

def get_file(name):
    path = Path(LOCAL_INPUT_DIR) / name if LOCAL_INPUT_DIR else INPUTS / name
    try:
        check_file(path, name)
    except (OSError, ValueError, yaml.YAMLError):
        if LOCAL_INPUT_DIR:
            raise RuntimeError(f"Missing or invalid attached file: {path}") from None
        partial = path.with_suffix(path.suffix + ".partial")
        partial.unlink(missing_ok=True)
        print(f"Downloading {name} ...", flush=True)
        try:
            result = gdown.download(id=DRIVE_IDS[name], output=str(partial), quiet=False)
            if not result:
                raise RuntimeError("Drive returned no file")
            check_file(partial, name)
            partial.replace(path)
        except Exception as exc:
            partial.unlink(missing_ok=True)
            raise RuntimeError(
                f"Could not download {name} ({type(exc).__name__}). "
                "Check that Internet is enabled and that the link opens in a signed-out browser. "
                "If Drive blocks or rate-limits it, use LOCAL_INPUT_DIR with a private Kaggle dataset."
            ) from exc
    print(f"Ready: {name} ({path.stat().st_size / 1e6:,.1f} MB)")
    return path

files = {name: get_file(name) for name in REQUIRED}
if DOWNLOAD_OPTIONAL_FILES:
    for name in ("last.pt", "config.yaml", "verifier.pt"):
        try:
            files[name] = get_file(name)
        except RuntimeError as exc:
            print(f"Optional file skipped: {exc}")

Downloading...
From (original): https://drive.google.com/uc?id=1okdPs196qHFB4iluMfihUApFyiUomtPJ
From (redirected): https://drive.google.com/uc?id=1okdPs196qHFB4iluMfihUApFyiUomtPJ&confirm=t&uuid=796857d7-bafd-48aa-a2db-7dcfd452b955
To: /tmp/fire-smoke-v4-cg_gi_zb/inputs/finetune_v4.tar.partial
100%|██████████| 1.23G/1.23G [00:16<00:00, 74.4MB/s]

Ready: finetune_v4.tar (1,232.1 MB)



Downloading...
From (original): https://drive.google.com/uc?id=1rkKXdfLgbbP7MwbFd0kbKSNM24_-7u5q
From (redirected): https://drive.google.com/uc?id=1rkKXdfLgbbP7MwbFd0kbKSNM24_-7u5q&confirm=t&uuid=af812585-6e00-40ee-942c-b9b60e8cca09
To: /tmp/fire-smoke-v4-cg_gi_zb/inputs/detector_v3_hardneg.pt.partial
100%|██████████| 44.0M/44.0M [00:00<00:00, 47.2MB/s]

Ready: detector_v3_hardneg.pt (44.0 MB)



Downloading...
From (original): https://drive.google.com/uc?id=12vuayUoms1Sm3aAZRjaUf9PSDRPgHbWV
From (redirected): https://drive.google.com/uc?id=12vuayUoms1Sm3aAZRjaUf9PSDRPgHbWV&confirm=t&uuid=7d230978-2065-47b3-a9a1-5e473d49f6be
To: /tmp/fire-smoke-v4-cg_gi_zb/inputs/detector_old.pt.partial
100%|██████████| 44.0M/44.0M [00:00<00:00, 50.8MB/s]

Ready: detector_old.pt (44.0 MB)


## 04. Validate the dataset splits

Extracts the dataset, checks image and label folders, and prepares the local split configurations.


In [4]:
# 4 | Extract the existing archive and validate its splits
from pathlib import PurePosixPath

def unpack_dataset(archive, root):
    with tarfile.open(archive, "r:*") as tf:
        members = tf.getmembers()
        seen = set()
        for m in members:
            p = PurePosixPath(m.name)
            if (not p.parts or p.parts[0] != "finetune_v4" or p.is_absolute()
                    or ".." in p.parts or not (m.isdir() or m.isfile())
                    or p in seen):
                raise ValueError(f"Unexpected archive entry: {m.name}")
            seen.add(p)
        needed = sum(m.size for m in members if m.isfile())
        if shutil.disk_usage(root).free < needed + 512 * 2**20:
            raise RuntimeError("Not enough local disk space to extract the dataset.")
        if hasattr(tarfile, "data_filter"):
            tf.extractall(path=root, members=members, filter="data")
        else:
            tf.extractall(path=root, members=members)  # Fresh staging directory; entries checked above.
    return root / "finetune_v4"

data_root = SCRATCH / "finetune_v4"
if not (data_root / ".verified").is_file():
    with tempfile.TemporaryDirectory(dir=SCRATCH) as staging:
        extracted = unpack_dataset(files["finetune_v4.tar"], Path(staging))
        audit = json.loads((extracted / "audit.json").read_text())
        for sp in ("train", "val", "test", "hard_val", "hard_test"):
            imgs = list((extracted / "images" / sp).glob("*.jpg"))
            expected = int(audit["splits"][sp]["total"])
            if not imgs or len(imgs) != expected:
                raise ValueError(f"{sp}: expected {expected} images, found {len(imgs)}")
            if any(not (extracted / "labels" / sp / (p.stem + ".txt")).is_file() for p in imgs):
                raise ValueError(f"{sp}: missing labels. Empty labels are allowed for verified negatives.")
            print(f"{sp:<10} {len(imgs):>6,} images")
        for fn in ("data.yaml", "hard_val.yaml"):
            src = yaml.safe_load((extracted / fn).read_text())
            if src["names"] not in (NAMES, list(NAMES.values())):
                raise ValueError(f"Unexpected class order in {fn}: {src['names']}")
        (extracted / ".verified").touch()
        extracted.rename(data_root)

DATA_YAML = data_root / "data.yaml"
HARD_YAML = data_root / "hard_val.yaml"
for p, val_split, test_split in ((DATA_YAML, "val", "test"), (HARD_YAML, "hard_val", "hard_test")):
    p.write_text(yaml.safe_dump({
        "path": str(data_root), "train": "images/train",
        "val": f"images/{val_split}", "test": f"images/{test_split}", "names": NAMES,
    }, sort_keys=False))

def model_for(path):
    m = YOLO(str(path))
    names = m.names if isinstance(m.names, dict) else dict(enumerate(m.names))
    if names != NAMES:
        raise ValueError(f"{path.name}: expected {NAMES}, got {names}")
    return m

for name in ("detector_v3_hardneg.pt", "detector_old.pt"):
    probe = model_for(files[name])
    del probe
print("Dataset and both checkpoints are ready. No training started yet.")

train      12,258 images
val         3,829 images
test        1,958 images
hard_val      869 images
hard_test     491 images
Dataset and both checkpoints are ready. No training started yet.


## 05. Continue detector fine-tuning

Loads the previous detector weights with a fresh optimizer and exports the best checkpoint as `detector_v4_kaggle.pt`.


In [5]:
# 5 | Further fine-tune from the Modal best weights (new optimizer)
from datetime import datetime, timezone

run_name = "t4_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
model = model_for(files["detector_v3_hardneg.pt"])
t0 = time.monotonic()
model.train(
    data=str(DATA_YAML), resume=False, epochs=EPOCHS, time=None,
    imgsz=IMGSZ, batch=BATCH, workers=WORKERS, device=DEVICE,
    optimizer="AdamW", lr0=LR, lrf=0.20, cos_lr=True, warmup_epochs=0.5,
    patience=0, hsv_h=0.005, hsv_s=0.50, hsv_v=0.30, degrees=0.0,
    translate=0.08, scale=0.35, fliplr=0.5, mosaic=0.25,
    mixup=0.0, close_mosaic=2, amp=True, cache=False, val=True,
    seed=SEED, deterministic=False, save=True, save_period=2, plots=True,
    project=str(RUNS), name=run_name, exist_ok=False,
)
run_dir = Path(model.trainer.save_dir)
best = run_dir / "weights" / "best.pt"
if not best.is_file():
    raise FileNotFoundError(f"Training did not save {best}")
# Keep the per-run weights and a convenient output file.
temp_export = EXPORT.with_suffix(".pt.partial")
shutil.copy2(best, temp_export)
temp_export.replace(EXPORT)
print(f"Candidate saved: {EXPORT}")
print(f"Run directory: {run_dir} | Training time: {(time.monotonic()-t0)/60:.1f} minutes")
del model
torch.cuda.empty_cache()

New https://pypi.org/project/ultralytics/8.4.212 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/fire-smoke-v4-cg_gi_zb/finetune_v4/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.005, hsv_s=0.5, hsv_v=0.3, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.2, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/20      7.22G      1.302      1.167    0.01469          2        800: 100% ━━━━━━━━━━━━ 3065/3065 4.8it/s 10:360.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 479/479 8.2it/s 58.5s0.2ss
                   all       3829       5275      0.753       0.64      0.729      0.454

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/20      7.22G      1.283      1.112    0.01434          3        800: 100% ━━━━━━━━━━━━ 3065/3065 4.8it/s 10:400.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 479/479 8.1it/s 59.0s0.2ss
                   all       3829       5275      0.741      0.647      0.727      0.449

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/20      7.22G      1.279      1.091    0.01417          1        800: 100% ━━━━━━━━━━━━ 3065/3065 4.8it/s 10:420.2ss
                

## 06. Compare all three detectors

Reports mAP on both validation splits, plus hard-validation false positives and per-class recall at a common confidence cutoff.


In [6]:
# 6 | Fair detector comparison (no training)
if not EXPORT.is_file():
    raise FileNotFoundError("Run training in Cell 5 first.")
CHECKPOINTS = {
    "original": files["detector_old.pt"],
    "modal_finetuned": files["detector_v3_hardneg.pt"],
    "kaggle_candidate": EXPORT,
}

def val_metrics(model, path, tag):
    v = model.val(data=str(path), split="val", imgsz=IMGSZ,
                  batch=EVAL_BATCH, device=DEVICE, workers=WORKERS,
                  verbose=False, plots=False, project=str(RUNS), name=tag)
    return {"mAP50": float(v.box.map50), "mAP50_95": float(v.box.map)}

def match_hits(gt, pred, iou_min=0.50):
    # gt = [class, xyxy]; pred = [class, confidence, xyxy] in descending confidence.
    matched = set()
    hits = np.zeros(2, dtype=int)
    for p in sorted(pred, key=lambda row: -row[1]):
        cls, _, x1, y1, x2, y2 = p
        candidates = [(i, g[1:]) for i, g in enumerate(gt)
                      if g[0] == cls and i not in matched]
        best_i, best_iou = None, iou_min
        for i, (a, b, c, d) in candidates:
            inter = max(0., min(x2, c) - max(x1, a)) * max(0., min(y2, d) - max(y1, b))
            union = (x2-x1)*(y2-y1) + (c-a)*(d-b) - inter
            score = inter / union if union > 0 else 0.
            if score >= best_iou:
                best_i, best_iou = i, score
        if best_i is not None:
            matched.add(best_i)
            hits[int(cls)] += 1
    return hits

def get_gt(image_path, shape):
    h, w = shape
    label = data_root / "labels" / "hard_val" / (Path(image_path).stem + ".txt")
    rows = []
    for line in label.read_text().splitlines():
        if line.strip():
            c, x, y, bw, bh = map(float, line.split())
            rows.append((int(c), (x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h))
    return rows

def fixed_conf_metrics(model, images):
    totals = np.zeros(2, dtype=int)
    hits = {t: np.zeros(2, dtype=int) for t in THRESHOLDS}
    fp = {t: {} for t in THRESHOLDS}
    for start in range(0, len(images), EVAL_BATCH):
        group = images[start:start+EVAL_BATCH]
        preds = model.predict([str(p) for p in group], imgsz=IMGSZ,
                              conf=min(THRESHOLDS), iou=0.50, max_det=50,
                              device=DEVICE, batch=EVAL_BATCH, verbose=False)
        if len(preds) != len(group):
            raise RuntimeError("Prediction count did not match the input count.")
        for r in preds:
            gt = get_gt(r.path, r.orig_shape)
            b = r.boxes
            pred = list(zip(b.cls.cpu().tolist(), b.conf.cpu().tolist(), *b.xyxy.cpu().numpy().T))
            for g in gt:
                totals[g[0]] += 1
            for t in THRESHOLDS:
                kept = [p for p in pred if p[1] >= t]
                if not gt:
                    fp[t][Path(r.path).name] = len(kept)
                else:
                    hits[t] += match_hits(gt, kept)
    report = {}
    for t in THRESHOLDS:
        n_neg = len(fp[t])
        fp_images = sum(v > 0 for v in fp[t].values())
        if not n_neg or not totals.all():
            raise RuntimeError("Hard validation needs negatives and examples of both classes.")
        report[t] = {
            "negatives": n_neg, "fp_images": fp_images, "fp_rate": fp_images / n_neg,
            "fp_boxes": sum(fp[t].values()),
            **{f"{label}_object_recall": float(hits[t][i] / totals[i]) for i, label in NAMES.items()},
            **{f"{label}_gt_boxes": int(totals[i]) for i, label in NAMES.items()},
        }
    return report, fp[PRIMARY_CONF]

hard_images = sorted((data_root / "images" / "hard_val").glob("*.jpg"))
rows, primary = [], {}
for tag, path in CHECKPOINTS.items():
    print(f"Evaluating {tag} ...", flush=True)
    orig = val_metrics(model_for(path), DATA_YAML, tag + "_original_val")
    hard = val_metrics(model_for(path), HARD_YAML, tag + "_hard_val")
    fixed, primary[tag] = fixed_conf_metrics(model_for(path), hard_images)
    torch.cuda.empty_cache()
    for t, stats in fixed.items():
        rows.append({
            "detector": tag, "confidence": t,
            **{f"original_val_{k}": v for k, v in orig.items()},
            **{f"hard_val_{k}": v for k, v in hard.items()},
            **stats,
        })

paired = {}
new = primary["kaggle_candidate"]
for tag in ("original", "modal_finetuned"):
    before = primary[tag]
    if new.keys() != before.keys():
        raise RuntimeError("Models did not see the same negative images.")
    paired[tag] = {
        "fixed": sorted(k for k in new if before[k] > 0 and new[k] == 0),
        "new_false_positives": sorted(k for k in new if before[k] == 0 and new[k] > 0),
    }
csv_path = WORKING / "comparison.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
(WORKING / "comparison.json").write_text(json.dumps({
    "metrics": rows, "paired_negatives_at_0.20": paired,
    "checkpoints": {tag: p.name for tag, p in CHECKPOINTS.items()},
    "imgsz": IMGSZ,
}, indent=2))

print(f"\nComparison at confidence {PRIMARY_CONF:.2f}")
for row in rows:
    if row["confidence"] == PRIMARY_CONF:
        print(f'{row["detector"]:<17} false images {row["fp_images"]:>3}/{row["negatives"]}'
              f' | fire recall {row["fire_object_recall"]:.1%} | smoke recall {row["smoke_object_recall"]:.1%}'
              f' | hard mAP50 {row["hard_val_mAP50"]:.1%}')
for tag, changes in paired.items():
    print(f'vs {tag}: fixed {len(changes["fixed"])} negative images,'
          f' introduced {len(changes["new_false_positives"])} new false-positive images')
print("Reports saved:", csv_path, "and comparison.json")

Evaluating original ...
Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 130 layers, 20,350,994 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1093.7±616.5 MB/s, size: 39.6 KB)
val: Scanning /tmp/fire-smoke-v4-cg_gi_zb/finetune_v4/labels/val.cache... 3829 images, 1615 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3829/3829 1.1Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 958/958 6.1it/s 2:370.1ss
                   all       3829       5275      0.767      0.684       0.76      0.482
Speed: 1.0ms preprocess, 37.5ms inference, 0.0ms loss, 0.7ms postprocess per image
Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 130 layers, 20,350,994 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1853.3±692.4 MB/s, size: 103.4 KB)
val: 